# Model Validation and Forecasting Protocol Audit

This notebook audits whether the earlier forecasting results are valid before they are used in any trustworthiness ranking. The central question is whether Naive persistence genuinely outperforms the neural models, or whether the apparent underperformance is caused by an implementation, alignment, scaling, target, or forecasting-protocol error.

No model should be promoted into the trustworthiness ranking until this audit passes.

In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Add, Dense, Dropout, Embedding, GlobalAveragePooling1D, Input, Layer, LayerNormalization, MultiHeadAttention
from tensorflow.keras.models import Model, Sequential

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_bitcoin_data
from src.preprocessing import prepare_daily_bitcoin_data
from src.metrics import mae, rmse, mape, smape

warnings.filterwarnings("ignore")
tf.random.set_seed(42)
np.random.seed(42)
pd.set_option("display.float_format", "{:.6f}".format)

## 1. Dataset and Split Verification

In [ ]:
data_path = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"
raw_df = load_bitcoin_data(data_path)
df_daily = prepare_daily_bitcoin_data(raw_df)
target = df_daily["Close"].asfreq("D").dropna().rename("Close")

split_idx = int(len(target) * 0.8)
train = target.iloc[:split_idx]
test = target.iloc[split_idx:]
y_test = test.copy()

train_test_overlap = train.index.intersection(test.index)

split_report = pd.DataFrame(
    {
        "Item": [
            "Complete series date range",
            "Training date range",
            "Test date range",
            "Training length",
            "Test length",
            "Train/test overlap",
        ],
        "Value": [
            f"{target.index.min()} to {target.index.max()}",
            f"{train.index.min()} to {train.index.max()}",
            f"{test.index.min()} to {test.index.max()}",
            len(train),
            len(test),
            len(train_test_overlap) > 0,
        ],
    }
)
split_report

## 2. Forecast Horizon Definition

In [ ]:
FORECAST_HORIZON = 1
LOOKBACK_LSTM = 30
LOOKBACK_IMPROVED_LSTM = 60
LOOKBACK_TRANSFORMER = 30

protocol_definition = pd.DataFrame(
    {
        "Protocol": ["A", "B"],
        "Name": ["Rolling one-step-ahead", "Recursive multi-step"],
        "Information available at each test date": [
            "All observed actual values up to the previous day are available.",
            "Only the final training window is available; future inputs are model predictions.",
        ],
        "Use of actual test values as future inputs": ["Allowed after each date is observed", "Not allowed"],
    }
)
protocol_definition

## 3. Sequence and Target Alignment

In [ ]:
def create_sequences(values, index, lookback):
    X, y, rows = [], [], []
    for i in range(lookback, len(values)):
        X.append(values[i - lookback : i])
        y.append(values[i])
        rows.append(
            {
                "input_window_start": index[i - lookback],
                "input_window_end": index[i - 1],
                "target_date": index[i],
                "last_input_value": float(values[i - 1, 0]),
                "target_value": float(values[i, 0]),
            }
        )
    return np.array(X), np.array(y), pd.DataFrame(rows)

scaler = MinMaxScaler(feature_range=(0, 1))
train_values = train.to_numpy().reshape(-1, 1)
test_values = test.to_numpy().reshape(-1, 1)
train_scaled = scaler.fit_transform(train_values)
test_scaled = scaler.transform(test_values)

X_train_lstm, y_train_lstm, train_sequence_map = create_sequences(train_scaled, train.index, LOOKBACK_LSTM)
combined_test_scaled = np.vstack([train_scaled[-LOOKBACK_LSTM:], test_scaled])
combined_test_index = train.index[-LOOKBACK_LSTM:].append(test.index)
X_test_lstm, y_test_lstm_scaled, test_sequence_map = create_sequences(combined_test_scaled, combined_test_index, LOOKBACK_LSTM)

test_sequence_map["sequence_ends_then_predicts_next_day"] = test_sequence_map["target_date"] == test_sequence_map["input_window_end"] + pd.Timedelta(days=1)
test_sequence_map.head(10)

In [ ]:
def alignment_checks(name, forecast, y_true):
    return {
        "Model": name,
        "Forecast length equals y_test length": len(forecast) == len(y_true),
        "Forecast index exactly equals y_test index": forecast.index.equals(y_true.index),
        "No missing values": not forecast.isna().any(),
        "No duplicated timestamps": not forecast.index.duplicated().any(),
        "No shifted-target symptom vs y[t+1]": not forecast.corr(y_true.shift(-1)) > forecast.corr(y_true),
    }

# Populated after forecasts are recreated below.
alignment_audit_rows = []

## 4. Scaling and Inverse-Scaling Audit

In [ ]:
roundtrip_train = scaler.inverse_transform(scaler.transform(train_values)).ravel()
roundtrip_test = scaler.inverse_transform(scaler.transform(test_values)).ravel()

scaling_audit = pd.DataFrame(
    {
        "Check": [
            "Scaler fitted only on training data",
            "scaler.data_min_",
            "scaler.data_max_",
            "Train scaled min/max",
            "Test scaled min/max",
            "Train inverse-transform roundtrip OK",
            "Test inverse-transform roundtrip OK",
            "Test values exceed training scaling range",
        ],
        "Result": [
            True,
            scaler.data_min_[0],
            scaler.data_max_[0],
            f"{train_scaled.min():.6f} to {train_scaled.max():.6f}",
            f"{test_scaled.min():.6f} to {test_scaled.max():.6f}",
            np.allclose(roundtrip_train, train_values.ravel()),
            np.allclose(roundtrip_test, test_values.ravel()),
            bool((test_scaled < 0).any() or (test_scaled > 1).any()),
        ],
    }
)
scaling_audit

## 5. Baseline Audit

In [ ]:
naive_forecast = target.shift(1).reindex(test.index).rename("Naive")
constant_last_train_forecast = pd.Series(train.iloc[-1], index=test.index, name="Constant Last Train Value")
moving_average_forecast = target.shift(1).rolling(window=7).mean().reindex(test.index).rename("7-Day Moving Average")

naive_expected = target.shift(1).reindex(test.index)
ma_expected = target.shift(1).rolling(window=7).mean().reindex(test.index)

baseline_audit = pd.DataFrame(
    {
        "Check": [
            "Naive equals actual[t-1]",
            "Naive differs from constant last-training-value forecast",
            "Moving average uses preceding seven observations only",
            "Moving average does not include target day",
        ],
        "Result": [
            naive_forecast.equals(naive_expected.rename("Naive")),
            not naive_forecast.equals(constant_last_train_forecast.rename("Naive")),
            moving_average_forecast.equals(ma_expected.rename("7-Day Moving Average")),
            moving_average_forecast.iloc[1:].ne(target.reindex(test.index).iloc[1:]).any(),
        ],
    }
)
baseline_audit

## 6. LSTM Audit

In [ ]:
def build_original_lstm(lookback=30):
    model = Sequential([Input(shape=(lookback, 1)), tf.keras.layers.LSTM(32), Dense(1)])
    model.compile(optimizer="adam", loss="mse")
    return model

lstm_model = build_original_lstm(LOOKBACK_LSTM)
lstm_callback = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
lstm_history = lstm_model.fit(
    X_train_lstm,
    y_train_lstm,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    callbacks=[lstm_callback],
    shuffle=False,
    verbose=0,
)
lstm_predictions_scaled = lstm_model.predict(X_test_lstm, verbose=0)
lstm_predictions = scaler.inverse_transform(lstm_predictions_scaled).ravel()
lstm_forecast = pd.Series(lstm_predictions, index=test.index, name="Original LSTM")

lstm_audit_table = pd.DataFrame(
    {
        "Check": [
            "Model input shape",
            "Model output shape",
            "X_train shape",
            "y_train shape",
            "X_test shape",
            "y_test shape",
            "Output layer activation is linear",
            "y_train is next value after each input window",
            "Final training loss",
            "Final validation loss",
        ],
        "Result": [
            str(lstm_model.input_shape),
            str(lstm_model.output_shape),
            str(X_train_lstm.shape),
            str(y_train_lstm.shape),
            str(X_test_lstm.shape),
            str(y_test_lstm_scaled.shape),
            lstm_model.layers[-1].activation.__name__ == "linear",
            bool((train_sequence_map["target_date"] == train_sequence_map["input_window_end"] + pd.Timedelta(days=1)).all()),
            lstm_history.history["loss"][-1],
            lstm_history.history["val_loss"][-1],
        ],
    }
)
lstm_model.summary()
lstm_audit_table

In [ ]:
def prediction_diagnostics(name, forecast, actual):
    aligned = pd.concat([actual.rename("actual"), forecast.rename("prediction")], axis=1).dropna()
    return pd.Series(
        {
            "prediction_min": aligned["prediction"].min(),
            "prediction_max": aligned["prediction"].max(),
            "prediction_mean": aligned["prediction"].mean(),
            "prediction_std": aligned["prediction"].std(),
            "actual_min": aligned["actual"].min(),
            "actual_max": aligned["actual"].max(),
            "actual_mean": aligned["actual"].mean(),
            "actual_std": aligned["actual"].std(),
            "prediction_daily_change_std": aligned["prediction"].diff().std(),
            "actual_daily_change_std": aligned["actual"].diff().std(),
            "correlation_with_actual": aligned["prediction"].corr(aligned["actual"]),
            "correlation_with_previous_day_actual": aligned["prediction"].corr(target.shift(1).reindex(aligned.index)),
            "percentage_unique_predictions": 100 * aligned["prediction"].nunique() / len(aligned),
        },
        name=name,
    )

lstm_first_20 = pd.concat([y_test.rename("Actual"), lstm_forecast.rename("Original LSTM")], axis=1).head(20)
lstm_diagnostics = prediction_diagnostics("Original LSTM", lstm_forecast, y_test)

display(lstm_first_20)
lstm_diagnostics.to_frame().T

## 7. Transformer Audit

In [ ]:
class PositionalEmbedding(Layer):
    def __init__(self, lookback, d_model, **kwargs):
        super().__init__(**kwargs)
        self.lookback = lookback
        self.d_model = d_model
        self.embedding = Embedding(input_dim=lookback, output_dim=d_model)

    def call(self, inputs):
        positions = tf.range(start=0, limit=self.lookback, delta=1)
        position_embedding = self.embedding(positions)
        return inputs + tf.expand_dims(position_embedding, axis=0)

    def get_config(self):
        config = super().get_config()
        config.update({"lookback": self.lookback, "d_model": self.d_model})
        return config


def build_corrected_transformer(lookback=30, d_model=64, num_heads=4, ff_dim=128, dropout=0.1, use_positional_embedding=True):
    inputs = Input(shape=(lookback, 1), name="price_window")
    x = Dense(d_model, name="input_projection")(inputs)

    if use_positional_embedding:
        x = PositionalEmbedding(lookback=lookback, d_model=d_model, name="positional_embedding")(x)

    attention_output = MultiHeadAttention(key_dim=d_model // num_heads, num_heads=num_heads, dropout=dropout, name="self_attention")(x, x)
    attention_output = Dropout(dropout)(attention_output)
    x = LayerNormalization(epsilon=1e-6, name="attention_norm")(x + attention_output)

    feed_forward = Dense(ff_dim, activation="relu", name="ffn_expand")(x)
    feed_forward = Dropout(dropout)(feed_forward)
    feed_forward = Dense(d_model, name="ffn_project")(feed_forward)
    x = LayerNormalization(epsilon=1e-6, name="ffn_norm")(x + feed_forward)

    x = GlobalAveragePooling1D(name="temporal_pooling")(x)
    x = Dropout(dropout)(x)
    x = Dense(32, activation="relu", name="prediction_hidden")(x)
    outputs = Dense(1, name="price_prediction")(x)
    model = Model(inputs, outputs, name="CorrectedTransformerAudit")
    model.compile(optimizer="adam", loss="mse")
    return model

X_train_tr, y_train_tr, train_transformer_map = create_sequences(train_scaled, train.index, LOOKBACK_TRANSFORMER)
combined_transformer_scaled = np.vstack([train_scaled[-LOOKBACK_TRANSFORMER:], test_scaled])
combined_transformer_index = train.index[-LOOKBACK_TRANSFORMER:].append(test.index)
X_test_tr, y_test_tr_scaled, test_transformer_map = create_sequences(combined_transformer_scaled, combined_transformer_index, LOOKBACK_TRANSFORMER)

transformer_model = build_corrected_transformer(LOOKBACK_TRANSFORMER)
transformer_callback = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
transformer_history = transformer_model.fit(
    X_train_tr,
    y_train_tr,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    callbacks=[transformer_callback],
    shuffle=False,
    verbose=0,
)
transformer_predictions_scaled = transformer_model.predict(X_test_tr, verbose=0)
transformer_predictions = scaler.inverse_transform(transformer_predictions_scaled).ravel()
corrected_transformer_forecast = pd.Series(transformer_predictions, index=test.index, name="Corrected Transformer")

layer_output_shapes = {layer.name: str(layer.output.shape) for layer in transformer_model.layers}
transformer_audit_table = pd.DataFrame(
    {
        "Check": [
            "LayerNormalization operates in D_MODEL space",
            "Explicit positional information is present",
            "Attention output connected before pooling/head",
            "Predictions are not constant",
            "Output layer activation is linear",
            "X_train shape",
            "y_train shape",
            "X_test shape",
            "y_test shape",
            "Final training loss",
            "Final validation loss",
        ],
        "Result": [
            all(layer_output_shapes[name].endswith(', 64)') for name in ["attention_norm", "ffn_norm"]),
            any("positional_embedding" in name for name in layer_output_shapes),
            "self_attention" in layer_output_shapes and "temporal_pooling" in layer_output_shapes,
            corrected_transformer_forecast.nunique() > 1,
            transformer_model.layers[-1].activation.__name__ == "linear",
            str(X_train_tr.shape),
            str(y_train_tr.shape),
            str(X_test_tr.shape),
            str(y_test_tr_scaled.shape),
            transformer_history.history["loss"][-1],
            transformer_history.history["val_loss"][-1],
        ],
    }
)
transformer_model.summary()
transformer_audit_table

In [ ]:
transformer_first_20 = pd.concat([y_test.rename("Actual"), corrected_transformer_forecast.rename("Corrected Transformer")], axis=1).head(20)
transformer_diagnostics = prediction_diagnostics("Corrected Transformer", corrected_transformer_forecast, y_test)

display(transformer_first_20)
transformer_diagnostics.to_frame().T

## 8. Rolling One-Step Evaluation

Protocol A: Rolling one-step-ahead forecasting. Each model predicts only the next day. The next sample may use the latest observed actual value. All models receive the same information at each test date.

In [ ]:
protocol_a_forecasts = {
    "Naive": naive_forecast,
    "7-Day Moving Average": moving_average_forecast,
    "Original LSTM": lstm_forecast,
    "Corrected Transformer": corrected_transformer_forecast,
}

protocol_a_alignment = pd.DataFrame(
    [alignment_checks(name, forecast, y_test) for name, forecast in protocol_a_forecasts.items()]
).set_index("Model")

protocol_a_metrics = pd.DataFrame(
    [
        {
            "Model": name,
            "MAE": mae(y_test, forecast),
            "RMSE": rmse(y_test, forecast),
            "MAPE": mape(y_test, forecast),
            "sMAPE": smape(y_test, forecast),
        }
        for name, forecast in protocol_a_forecasts.items()
    ]
).set_index("Model").sort_values("RMSE")

display(protocol_a_alignment)
protocol_a_metrics

## 9. Recursive Multi-Step Evaluation

Protocol B: Recursive multi-step forecasting. Start only from the final training window, predict the first test value, feed predictions back into the model recursively, and do not use actual test values as future inputs.

In [ ]:
def recursive_lstm_forecast(model, scaler, train_series, forecast_index, lookback, name):
    history_scaled = list(scaler.transform(train_series.to_numpy().reshape(-1, 1)).ravel())
    predictions = []
    for _ in forecast_index:
        window = np.array(history_scaled[-lookback:]).reshape(1, lookback, 1)
        pred_scaled = float(model.predict(window, verbose=0).ravel()[0])
        history_scaled.append(pred_scaled)
        predictions.append(pred_scaled)
    predictions = scaler.inverse_transform(np.array(predictions).reshape(-1, 1)).ravel()
    return pd.Series(predictions, index=forecast_index, name=name)

recursive_lstm = recursive_lstm_forecast(lstm_model, scaler, train, test.index, LOOKBACK_LSTM, "Original LSTM Recursive")
recursive_transformer = recursive_lstm_forecast(transformer_model, scaler, train, test.index, LOOKBACK_TRANSFORMER, "Corrected Transformer Recursive")

protocol_b_forecasts = {
    "Original LSTM Recursive": recursive_lstm,
    "Corrected Transformer Recursive": recursive_transformer,
}

protocol_b_metrics = pd.DataFrame(
    [
        {
            "Model": name,
            "MAE": mae(y_test, forecast),
            "RMSE": rmse(y_test, forecast),
            "MAPE": mape(y_test, forecast),
            "sMAPE": smape(y_test, forecast),
        }
        for name, forecast in protocol_b_forecasts.items()
    ]
).set_index("Model").sort_values("RMSE")
protocol_b_metrics

## 10. Fair Model Comparison

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

def build_delta_lstm(lookback=30):
    model = Sequential([Input(shape=(lookback, 1)), tf.keras.layers.LSTM(32), Dense(1)])
    model.compile(optimizer="adam", loss="mse")
    return model

log_returns = np.log(target / target.shift(1)).dropna().rename("log_return")
return_train = log_returns.reindex(train.index).dropna()
return_test_index = test.index
return_scaler = MinMaxScaler(feature_range=(-1, 1))
return_train_scaled = return_scaler.fit_transform(return_train.to_numpy().reshape(-1, 1))

X_return_train, y_return_train, _ = create_sequences(return_train_scaled, return_train.index, LOOKBACK_LSTM)
delta_model = build_delta_lstm(LOOKBACK_LSTM)
delta_history = delta_model.fit(
    X_return_train,
    y_return_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    callbacks=[EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)],
    shuffle=False,
    verbose=0,
)

known_returns = list(return_scaler.transform(return_train.to_numpy().reshape(-1, 1)).ravel())
price_history = list(train.to_numpy())
delta_price_predictions = []
for date in return_test_index:
    window = np.array(known_returns[-LOOKBACK_LSTM:]).reshape(1, LOOKBACK_LSTM, 1)
    pred_return_scaled = float(delta_model.predict(window, verbose=0).ravel()[0])
    pred_return = float(return_scaler.inverse_transform([[pred_return_scaled]])[0, 0])
    next_price = price_history[-1] * np.exp(pred_return)
    delta_price_predictions.append(next_price)
    actual_return = np.log(target.loc[date] / price_history[-1])
    known_returns.append(float(return_scaler.transform([[actual_return]])[0, 0]))
    price_history.append(target.loc[date])

persistence_enhanced_lstm = pd.Series(delta_price_predictions, index=test.index, name="Persistence-Enhanced LSTM")

fair_comparison_forecasts = {
    **protocol_a_forecasts,
    "Persistence-Enhanced LSTM": persistence_enhanced_lstm,
}
fair_comparison_metrics = pd.DataFrame(
    [
        {
            "Model": name,
            "MAE": mae(y_test, forecast),
            "RMSE": rmse(y_test, forecast),
            "MAPE": mape(y_test, forecast),
            "sMAPE": smape(y_test, forecast),
            "Prediction Std": forecast.std(),
            "Actual Std": y_test.std(),
            "Prediction Change Std": forecast.diff().std(),
            "Actual Change Std": y_test.diff().std(),
        }
        for name, forecast in fair_comparison_forecasts.items()
    ]
).set_index("Model").sort_values("RMSE")
fair_comparison_metrics

### Persistence-Enhanced LSTM Forecast Artifact

The exact fixed-seed rolling one-step forecast vector is saved for downstream significance testing.

In [ ]:
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(exist_ok=True)

persistence_artifact = pd.DataFrame(
    {
        "Timestamp": persistence_enhanced_lstm.index,
        "Persistence_Enhanced_LSTM": persistence_enhanced_lstm.to_numpy(dtype=float),
    }
)

assert len(persistence_artifact) == 1061
assert pd.DatetimeIndex(persistence_artifact["Timestamp"]).equals(y_test.index)
assert persistence_artifact["Timestamp"].is_unique
assert persistence_artifact["Timestamp"].is_monotonic_increasing
assert not persistence_artifact.isna().any().any()
assert np.isfinite(persistence_artifact["Persistence_Enhanced_LSTM"].to_numpy(dtype=float)).all()

persistence_artifact_path = results_dir / "persistence_enhanced_lstm_forecast.csv"
persistence_artifact.to_csv(persistence_artifact_path, index=False)

persistence_artifact_metrics = pd.DataFrame(
    [
        {
            "Model": "Persistence-Enhanced LSTM",
            "MAE": mae(y_test, persistence_enhanced_lstm),
            "RMSE": rmse(y_test, persistence_enhanced_lstm),
            "MAPE": mape(y_test, persistence_enhanced_lstm),
            "sMAPE": smape(y_test, persistence_enhanced_lstm),
        }
    ]
).set_index("Model")

print(f"Saved {persistence_artifact_path}")
print(f"Shape: {persistence_artifact.shape}")
print("Neural-network forecasts were saved from a fixed-seed deterministic run. Earlier aggregate LSTM metrics came from a different nondeterministic training run and are not used for significance testing.")
persistence_artifact_metrics


In [ ]:
plot_sets = {
    "Full Test Period": y_test.index,
    "First 90 Test Days": y_test.index[:90],
    "Final 90 Test Days": y_test.index[-90:],
}

for title, idx in plot_sets.items():
    fig, ax = plt.subplots(figsize=(12, 5))
    y_test.reindex(idx).plot(ax=ax, label="Actual", linewidth=2)
    for name, forecast in fair_comparison_forecasts.items():
        forecast.reindex(idx).plot(ax=ax, label=name, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel("Bitcoin Close")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
y_test.diff().plot(ax=ax, label="Actual change", linewidth=2)
for name, forecast in fair_comparison_forecasts.items():
    forecast.diff().plot(ax=ax, label=f"{name} change", alpha=0.75)
ax.set_title("Actual vs Predicted Daily Changes")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
y_test.plot(kind="hist", bins=40, alpha=0.4, ax=ax, label="Actual")
for name, forecast in fair_comparison_forecasts.items():
    forecast.plot(kind="hist", bins=40, alpha=0.25, ax=ax, label=name)
ax.set_title("Prediction Distribution vs Actual Distribution")
ax.legend()
plt.tight_layout()
plt.show()

## 11. Final Diagnosis

In [ ]:
diagnosis = pd.DataFrame(
    {
        "Audit Area": [
            "Implementation validity",
            "Forecasting-protocol validity",
            "Model collapse",
            "Over-smoothing",
            "Range compression",
            "Data leakage",
            "Target misalignment",
            "Scaling extrapolation",
        ],
        "Evidence to Review": [
            "Model summaries, linear output activations, shape checks, alignment checks.",
            "Protocol A and Protocol B metrics are reported separately.",
            "Prediction uniqueness, prediction std, and constant-prediction checks.",
            "Prediction daily-change std versus actual daily-change std.",
            "Prediction min/max/std versus actual min/max/std.",
            "Naive and moving-average checks use only prior actual observations; Protocol B forbids test actual feedback.",
            "Sequence maps confirm input window ending on t predicts t+1.",
            "Scaler is fit on train only; test scaled min/max shows extrapolation beyond training range.",
        ],
        "Pass Criteria": [
            "All checks pass or failures are explicitly documented.",
            "No cross-protocol ranking is presented as a single fair leaderboard.",
            "Predictions are not constant except intentionally constant baseline.",
            "Neural change variance is not near zero relative to actual changes.",
            "Forecast range is not implausibly compressed without explanation.",
            "No forecast uses target-day information.",
            "All target dates are exactly one day after window end for one-step models.",
            "Inverse scaling roundtrip passes and extrapolation is acknowledged.",
        ],
    }
)
diagnosis

Final reporting guidance:

- Do not claim that advanced models should automatically beat Naive.
- Treat Naive outperformance as plausible for noisy Bitcoin close prices unless the audit finds leakage, misalignment, or protocol inconsistency.
- Do not use these models in the trustworthiness ranking until this notebook has been run and the diagnosis table is reviewed.
- If neural forecasts remain flat after alignment and scaling checks pass, the likely diagnosis is model/target mismatch or over-smoothing from raw price-level modelling rather than a simple implementation error.